In [ ]:
%%configure

{
    "vCores":
    {
        "parameterName": "pipelinecore",
        "defaultValue": 2
    }
}


In [ ]:
# THE SCHEDULED FORM OF ONE CI LEG. This notebook runs the SAME scripts the GitHub workflow
# runs (.github/scripts/provision.py, download_aemo.py, .github/scripts/run_dbt.py) from
# the copy of the repo deploy.py put in the `dbt` lakehouse's Files/dbt, on the engine the
# deploy_config Variable Library names. Nothing dbt-shaped lives here: change the scripts,
# redeploy.
#
# Do not schedule it alongside a pipeline.yml run: both land into dbt_landing, and the archive
# log is rewritten with a delete-then-copy.
import os
import shutil
import subprocess
import sys

import notebookutils

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
engine = vl.dbt_target
for k in ("process_limit", "download_limit", "daily_download_limit"):
    os.environ[k] = getattr(vl, k)
ws_id = notebookutils.runtime.context["currentWorkspaceId"]
os.environ["FABRIC_WORKSPACE_ID"] = ws_id

# The ~135 GiB work disk, not the 19 GiB /tmp overlay: the probe duckrun's run_python uses.
# run_dbt.py derives DuckDB's spill dir from TMPDIR when nothing else set it.
scratch = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
os.environ["TMPDIR"] = os.path.join(scratch, "duckrun_tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# `dbt` is provision.py's DATA_LAKEHOUSE; deploy.py copies the git-tracked repo into its Files/dbt.
# Mounted, then copied to the work disk: dbt's target/ and DuckDB's spill must not land on OneLake.
repo = os.path.join(scratch, "dbt")
lh_id = notebookutils.lakehouse.get("dbt", ws_id).get("id")
notebookutils.fs.mount(
    f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}", "/lh", {"fileCacheTimeout": 0}
)
shutil.copytree(f"{notebookutils.fs.getMountPath('/lh')}/Files/dbt", repo, dirs_exist_ok=True)
os.chdir(repo)
sys.path.insert(0, f"{repo}/.github/scripts")
import remote_dbt  # the CI leg's pip list and token hooks, reused verbatim

assert engine in remote_dbt.ENGINES, f"dbt_target={engine!r}: this notebook runs one of {remote_dbt.ENGINES}"
# Into this session's environment; dbt runs in subprocesses below, so no restartPython (which
# would also drop the tokens and the env exported here).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *remote_dbt.pip_args(engine)], check=True)


In [ ]:
# provision -> land -> build || retry -> fingerprint, each a subprocess of this kernel.
prov = subprocess.run([sys.executable, ".github/scripts/provision.py", engine],
                      cwd=repo, text=True, capture_output=True)
sys.stderr.write(prov.stderr)
if prov.returncode:
    raise RuntimeError(f"provision.py {engine} failed (exit {prov.returncode})")
# KEY=value lines, split once: the DuckLake DSN carries `=` of its own.
os.environ.update(dict(line.split("=", 1) for line in prov.stdout.splitlines() if "=" in line))

subprocess.run([sys.executable, "download_aemo.py"], cwd=repo, check=True)

# Tokens, minted here from notebookutils exactly as remote_dbt.py's setup hook does for CI.
if remote_dbt.SETUP[engine]:
    exec(remote_dbt.SETUP[engine])

rc = subprocess.run([sys.executable, ".github/scripts/run_dbt.py", engine], cwd=repo).returncode
if rc:
    # Raise so a scheduled Fabric run goes red instead of silently reporting success.
    raise RuntimeError(f"dbt {engine} failed (exit {rc}); see the log above")
